# A Stationary IMU

In this example we consider an IMU lying stationary on a table, and consider whether we can accurately estimate its orientation. Unfortunately, using only the IMU measurements, both yaw and velocity are unobservable, so we *must* provide additional information in order to get an answer.

In [26]:
import gtsam
import numpy as np
import graphviz
Iso = gtsam.noiseModel.Isotropic

In [ ]:
class show(graphviz.Source):
    """
    Display an object with a dot method as a graph.
    NOTE: This needs "conda install python-graphviz
    """

    def __init__(self, obj, *args, **kwargs):
        super().__init__(obj.dot(*args, **kwargs), engine="neato")

# Simulating an IMU

We first set up a simulated IMU using the `ScenarioRunner` class. It's not always straightforward how to correctly deal with coordinate frames and gravity. In particular, we have to make a choice about coordinate frame conventions. GTSAM supports two common choices for the **navigation frame**, which is typically chosen as a (local plane](https://en.wikipedia.org/wiki/Local_tangent_plane_coordinates) tangent to the earth surface, fixed at some origin nearby the vehicle:

- a Z-down navigation frame, such as North-East-Down (NED): gravity points along positive Z-axis
- a Z-up navigation frame, such as East-North-Up (ENU): gravity points along negative Z-axis

It is easier to think in an ENU frame, so below we create a `gtsam.PreintegrationParams` instance that uses that convention. We also provide a local gravity constant, which in Atlanta would be around 9.8.


In [28]:
# MakeSharedU chooses Z-up!
noise_parameters = gtsam.PreintegrationParams.MakeSharedU(9.8)

In [2]:
# Set up an IMU simulator at rest
wb = np.array([0, 0, 0])
vb = np.array([0, 0, 0])
scenario = gtsam.ConstantTwistScenario(wb, vb)

# Create noise parameters
kGyroSigma = np.radians(0.5) / 60  # 0.5 degree ARW
kAccelSigma = 0.1 / 60  # 10 cm VRW
I3 = np.identity(3, float)
noise_parameters.setGyroscopeCovariance(kGyroSigma**2 * I3)
noise_parameters.setAccelerometerCovariance(kAccelSigma**2 * I3)
noise_parameters.setIntegrationCovariance(0.0000001**2 * I3) # ignore this for now

# Set the sample time
dt = 0.1 # 10 Hz

# Create zero bias parameters
accBias = np.array([0, 0, 0]) # bias in accelerometer
gyroBias = np.array([0, 0, 0]) # bias in gyroscope
bias = gtsam.imuBias.ConstantBias(accBias, gyroBias)

# Instantiate IMU simulation class as `runner`
runner = gtsam.ScenarioRunner(scenario, noise_parameters, dt, bias)


In [3]:
# Define key naming scheme for GTSAM
X = gtsam.symbol_shorthand.X # for poses
V = gtsam.symbol_shorthand.V # for velocities
B0 = gtsam.symbol('b', 0) # for bias


In [4]:
# Create graph with one IMU factor and priors on pose, velocity, and bias
imu_graph = gtsam.NonlinearFactorGraph()
state = scenario.navState(0)
imu_graph.addPriorPose3(X(0), state.pose(), Iso.Sigma(6, 0.1))  # 6 DOF
imu_graph.addPriorVector(V(0), state.velocity(), Iso.Sigma(3, 0.1))  # 3 DOF
imu_graph.addPriorConstantBias(B0, bias, Iso.Sigma(6, 0.1))  # 6 DOF

# Add a PIM with 10 measurements
pim = gtsam.PreintegratedImuMeasurements(noise_parameters, bias)
for j in range(10):
    t = dt * j
    measuredOmega = runner.measuredAngularVelocity(t)
    measuredAcc = runner.measuredSpecificForce(t)
    pim.integrateMeasurement(measuredAcc, measuredOmega, dt)

factor = gtsam.ImuFactor(X(0), V(0), X(1), V(1), B0, pim) # 9 DOF
imu_graph.push_back(factor)

In [5]:
# Create values with initial state
values = gtsam.Values()
state = scenario.navState(0)
values.insert(X(0), state.pose())
values.insert(V(0), state.velocity())

state = scenario.navState(1)
values.insert(X(1), state.pose())
values.insert(V(1), state.velocity())

# Add bias, as we will need it below
values.insert(B0, bias)

In [ ]:
#| caption: Factor graph with one IMU factor and priors
#| label: fig:imu_graph
show(imu_graph, gtsam.Values())


In [7]:
linear_graph = imu_graph.linearize(values)

In [ ]:
A,b = linear_graph.jacobian()

In [ ]:
print(A.shape)